In [1]:
import requests
# import pandas as pd

from datetime import datetime, timedelta


In [2]:

import sys
import os

root = os.path.abspath(os.path.join(os.getcwd(), "../"))

if root not in sys.path:
    sys.path.insert(0, root)

print(root)

/home/hilaneto/Jhan/Desenvolvimento/Python/Projetos/AtualizaIndicador


In [3]:

# Datas -------------------------------------------------------
hoje = datetime.now()

data_inicial = (hoje - timedelta(days=62)).strftime("%d/%m/%Y")
data_final = hoje.strftime("%d/%m/%Y")

print(f"Período: {data_inicial} até {data_final}")


Período: 19/06/2026 até 20/08/2026


In [ ]:

# https://api.bcb.gov.br/dados/serie/bcdata.sgs.10844/dados?formato=json&dataInicial=25/06/2026&dataFinal=18/08/2026

# API Banco Central ------------------------------------------
url = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.10844/dados"

parametros = { "formato": "json", "dataInicial": data_inicial, "dataFinal": data_final }

resposta = requests.get(url, params=parametros, timeout=10)
resposta.raise_for_status()
dados = resposta.json() # lista de dicionários

# Percorrer a lista de dicionário
for ipca in dados:
    print(ipca)


In [ ]:
for ipca in dados:
    print(ipca["data"], ipca["valor"])


In [ ]:

from database.conexao import conectar
from indicadores.ipca import Ipca
from decimal import Decimal

dados_ipca = []

with conectar():
    
    for registro in dados:
        ipca = {"indice": Decimal(registro["valor"]),
                "status": True,
                "dt_referencia": datetime.strptime(registro["data"],"%d/%m/%Y").date(),
                "dt_atualizacao": datetime.now()}
        dados_ipca.append(ipca)
        
        Ipca.insert(**ipca).on_conflict(
        conflict_target=[Ipca.dt_referencia],
        update={Ipca.indice: ipca["indice"],
                Ipca.status: ipca["status"],
                Ipca.dt_atualizacao: ipca["dt_atualizacao"]}).execute()


In [8]:

from database.conexao import conectar
from indicadores.ipca import Ipca

with conectar():
    dados_ipca = Ipca.buscar()
    print(dados_ipca)


[{'data': '01/06/2026', 'valor': '0.33'}, {'data': '01/07/2026', 'valor': '0.54'}]


In [7]:

from database.conexao import conectar
from indicadores.ipca import Ipca

aa = Ipca.atualizar_ipca()

print(aa)

None
